In [ ]:
#| default_exp cli.ingest

In [ ]:
#| export
from fastcore.script import *
from typing import Optional, Literal
from fastcore.style import S
import sys
import importlib
from importlib import import_module
from pkgutil import iter_modules
import marisco.handlers

In [ ]:
#| export
def import_handler(handler_name:str,  # Name of the handler module
                   fn_name:str='encode'  # Name of the function to import
                  )->callable:  # The requested function
    "Import `fn_name` from module `handler_name`"
    try:
        handler = import_module(handler_name)
        return getattr(handler, fn_name)
    except (ImportError, AttributeError): 
        print(f"Failed to import function: {fn_name}")

In [ ]:
#| export
def handlers() -> dict:
    "Dict of available handlers and their status"
    return {m.name:getattr(import_module(f'marisco.handlers.{m.name}'), 'status', 'Active') for m in iter_modules(marisco.handlers.__path__)}

In [ ]:
#| eval: false
handlers()

{'fram_strait2025': 'Active',
 'geotraces': 'Active',
 'helcom': 'Active',
 'jois': 'Active',
 'maris_legacy': 'Active',
 'ospar': 'Under refactoring',
 'tepco': 'Under refactoring'}

In [ ]:
#| export
@call_parse
def main(
    ds: str,  # Name of the dataset to encode as NetCDF4 (e.g. helcom, geotraces, maris_legacy)
    dest: str, # Output path: file path for handlers that write one file, or folder path for handlers that write several (e.g. maris_legacy)
    src: Optional[str] = None,  # Optional path to local input data; only needed by handlers that don't fetch data online
    ref_ids: Optional[str] = None,  # Comma-separated MARIS reference IDs, e.g. "16,30"; only used by handlers that support them
) -> None:
    "Convert a marine radioactivity dataset to MARIS NetCDF4 format."
    hs = handlers()
    if ds not in hs:
        print(S.red(f"Invalid handler name: {ds}. Available handlers: {', '.join(hs)}"))
        sys.exit(1)
    if hs[ds] != 'Active':
        print(S.yellow(f"Warning: {ds} is {hs[ds]}"))
        sys.exit(1)
    encode = import_handler(f'marisco.handlers.{ds}')
    print(f'Encoding: {ds} ...')
    encode(dest=dest, src=src, **({'ref_ids': ref_ids} if ref_ids else {}))
    print(S.green(f'Done: {ds} encoded to {dest}'))

## Usage

`marisco-ingest` converts a provider dataset to the MARIS NetCDF4 format. The handler is selected by name, and each handler decides whether it reads local files (`src`) or fetches data online.

- Single-file output (helcom, geotraces, tepco, ...): `marisco-ingest helcom output/100-HELCOM-MORS-2024.nc`

- Folder output with a `ref_ids` subset (maris_legacy): `marisco-ingest maris_legacy ~/output --src ~/data/maris/dump.txt --ref_ids "16,30"`

Run `marisco-ingest` with an invalid handler name to list all available handlers.